# Performance Benchmarking: Building Reproducible Spark Benchmark Suites

**Module**: 10_performance_and_optimization  
**Notebook**: 07_performance_benchmarking.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How to design reproducible benchmarks in PySpark: warming up kernels, measuring wall-clock time, collecting Spark UI metrics, and comparing implementations.
- **Why does this topic exist?**: Optimization claims must be supported by empirical data, not guesswork.
- **What problem does it solve?**: Benchmarking un-warmed jobs or confusing client overhead with cluster compute time.
- **Where is it used in real-world Data Engineering?**: Validating migration of pipelines from on-premises to cloud.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Benchmarking_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")

def benchmark_op(name, func, runs=3):
    # Warmup
    func()
    times = []
    for _ in range(runs):
        t0 = time.time()
        func()
        times.append(time.time() - t0)
    avg_time = sum(times) / len(times)
    print(f"Benchmark '{name}': Avg = {avg_time:.4f}s across {runs} runs (Min: {min(times):.4f}s)")
    return avg_time

benchmark_op("DataFrame Count", lambda: orders.count())
benchmark_op("DataFrame Distinct Status", lambda: orders.select("status").distinct().count())

---

## Experiment: Comparing Warmup vs Cold Run Latency

- **1. What are we testing?**: Demonstrate that the first run is slower due to class loading, JVM JIT compilation, and connection establishment.
- **2. What do we expect?**: First run is 2x slower than subsequent runs.

In [ ]:
t0 = time.time(); orders.groupBy("channel").count().count(); t_cold = time.time() - t0
t0 = time.time(); orders.groupBy("channel").count().count(); t_warm = time.time() - t0
print(f"Cold Run: {t_cold:.4f}s vs Warm Run: {t_warm:.4f}s")

**3. What happened?**: Cold run was noticeably slower.  
**4. Why did it happen?**: JVM JIT compilation, executor connection handshake, and classloading occurred on first run.  
**5. Practical Engineering Lesson**: Always run at least one warmup pass before recording benchmark measurements.

---

## Practical Exercises

### Exercise 1 (Beginner): Benchmark Join
**Problem Statement**: Benchmark join between orders and customers.  
**Input / Context**: orders, customers

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Measure Shuffle Read
**Problem Statement**: How to inspect shuffle read metrics via statusTracker.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): A/B Testing Framework
**Problem Statement**: Design a function to compare two DataFrames execution times.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Benchmark Join):
```python
custs = spark.read.option('header', True).csv('/opt/spark-data/retail/customers.csv')
benchmark_op('Join', lambda: orders.join(custs,'customer_id').count())
```

#### Solution 2 (Measure Shuffle Read):
```python
print('Query TaskContext or statusTracker for stage metrics.')
```

#### Solution 3 (A/B Testing Framework):
```python
print('Run both functions with multiple iterations and statistical t-test.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Warmup Rule**: Always warm up the JVM before recording metrics.
- **Action Isolation**: Measure wall clock exclusively around the terminal Action.

### Essential Technical Terms
- `JIT Compilation`: JVM runtime bytecode compilation to native machine code.
- `Cold Run`: First execution before caches/compilation settle.

### Common Engineering Mistakes
- **Measuring transformation lines**: Transformations are lazy; timing them measures AST creation, not compute.

### Interview Questions on This Topic
- **Q: Why is the first execution of a Spark query slower than subsequent runs?**
  - *A*: Because of initial classloading, JVM JIT compilation, executor thread initialization, and storage metadata caching.

### What to Study Next
Module 10 complete! Proceed to Module 11: Fault Tolerance and Reliability.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')